<a href="https://colab.research.google.com/github/ClaraMourgueDAlgue/QM2/blob/main/Copy_of_qf209_a4_person2_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# QF209 A4 Smart Predict-then-Optimize | Sprint 2 — Person 2 baseline
**Owner: Person 2 · OLS/ridge forecasts + mean–variance optimisation + baseline diagnosis**

Inputs: `train_panel.csv`, `validation_panel.csv`, `validation_benchmark_metrics.csv`, `iteration_log.csv` from Person 1's package (put them next to this notebook, or upload them in Colab). **No test labels are used. The test set stays locked.**

Design choices (all fixed before looking at validation):
- **Forecast:** one pooled linear model for all 20 ETFs, using the 9 features z-scored on the fitting window. α = 0 is OLS; α > 0 is ridge (intercept not penalised). Fitted once on all 2010–2018 training weeks, then frozen for validation (no rolling retrain in the baseline).
- **Risk:** sample covariance of the last 156 weekly returns (`ret_1w` up to and including the decision Friday, so only information known at t).
- **Optimiser:** max μᵀw − ½γ wᵀΣw subject to Σw = 1, Σ|w| ≤ 1.5, |wᵢ| ≤ 1. Solved with scipy SLSQP using the split w = u − v; no cvxpy needed.
- **Tuning on the 4 internal folds only:** α by lowest fold MSE (the "accuracy" step), then γ by highest mean fold net Sharpe given that α.
- **Scoring:** Person 1's `evaluate_portfolio`, copied verbatim (5 bp, rf = 0).

In [6]:
# Colab: upload the CSVs first, e.g.
# from google.colab import files; files.upload()
import os; os.makedirs("fig", exist_ok=True)


## 1. Core functions (data loading, shared backtester, forecaster, covariance, optimiser)

In [7]:
"""QF209 A4 - Person 2: OLS/ridge predict-then-optimise baseline.
Uses only train_panel.csv and validation_panel.csv (no test labels).
"""
import numpy as np, pandas as pd
from scipy.optimize import minimize

TICKERS = ['SPY','XLK','XLF','XLE','XLV','XLI','XLY','XLP','XLU','XLB',
           'QQQ','IWM','EFA','EEM','TLT','IEF','LQD','HYG','GLD','VNQ']
FEATURES = ['ret_1w','mom_4w','mom_12w','mom_26w','vol_12w','vol_26w',
            'spy_mom_4w','spy_mom_12w','spy_vol_12w']
MAX_GROSS, MAX_W, COST_BPS = 1.5, 1.0, 5.0
COV_WINDOW = 156  # weeks of past realised returns for the covariance estimate


def load_panels(train_path='train_panel.csv', val_path='validation_panel.csv'):
    out = {}
    for name, p in [('train', train_path), ('validation', val_path)]:
        df = pd.read_csv(p, parse_dates=['decision_date', 'target_end_date'])
        out[name] = df.set_index(['decision_date', 'ticker']).sort_index()
    panel = pd.concat([out['train'], out['validation']]).sort_index()
    split_dates = {k: v.index.get_level_values(0).unique().sort_values() for k, v in out.items()}
    targets = panel['target_return'].unstack('ticker')[TICKERS]
    # ret_1w at decision date t is the return of the week ENDING at t -> known at t
    past_returns = panel['ret_1w'].unstack('ticker')[TICKERS]
    return panel, split_dates, targets, past_returns


# ---------------- shared backtester (verbatim from Person 1's notebook) ----------------
def evaluate_portfolio(weights, forward_returns, predictions=None,
                       cost_bps=5.0, rf_annual=0.0,
                       allow_short=True, max_gross=1.5, max_weight_abs=1.0):
    if not isinstance(weights, pd.DataFrame) or not isinstance(forward_returns, pd.DataFrame):
        raise TypeError('weights and forward_returns must be pandas DataFrames')
    if weights.empty: raise ValueError('Weights are empty.')
    tickers = list(forward_returns.columns)
    if set(weights.columns) != set(tickers):
        raise ValueError('Weight columns must exactly equal forward return tickers.')
    if not weights.index.is_unique or not weights.index.is_monotonic_increasing:
        raise ValueError('Weight dates must be unique and increasing.')
    if any(date not in forward_returns.index for date in weights.index):
        raise ValueError('A weight date is missing from forward_returns.')
    if not (weights.index.to_series().diff().dropna() == pd.Timedelta(days=7)).all():
        raise ValueError('Weight dates must be consecutive Fridays; otherwise missed weekly P&L is hidden.')
    wdf = weights.reindex(columns=tickers).astype(float)
    rdf = forward_returns.reindex(index=wdf.index, columns=tickers).astype(float)
    if not (np.isfinite(wdf.to_numpy()).all() and np.isfinite(rdf.to_numpy()).all()):
        raise ValueError('Nonfinite weights or actual returns.')
    if not np.allclose(wdf.sum(axis=1).values, 1.0, atol=1e-6):
        raise ValueError('Every portfolio row must sum to 1.0 (fully invested).')
    if (wdf.abs().sum(axis=1) > max_gross + 1e-6).any():
        raise ValueError('Weights breach the gross leverage limit.')
    if (wdf.abs() > max_weight_abs + 1e-6).any().any():
        raise ValueError('Weights breach per-asset absolute-weight limit.')
    if not allow_short and (wdf < -1e-8).any().any():
        raise ValueError('Negative weights not allowed in long-only experiment.')
    if not 0 <= cost_bps <= 10000: raise ValueError('Invalid cost in basis points.')
    pre_trade = np.zeros(len(tickers), dtype=float)
    records = []
    fee_rate = cost_bps / 10000.0
    for date in wdf.index:
        target = wdf.loc[date].to_numpy(dtype=float)
        r = rdf.loc[date].to_numpy(dtype=float)
        traded_notional = float(np.abs(target - pre_trade).sum())
        cost_fraction = fee_rate * traded_notional
        gross_return = float(np.dot(target, r))
        if 1 + gross_return <= 0 or cost_fraction >= 1:
            raise ValueError(f'Invalid portfolio wealth factor at {date}.')
        net_return = (1 - cost_fraction) * (1 + gross_return) - 1
        records.append({'decision_date': date, 'gross_return': gross_return,
                        'net_return': net_return, 'turnover': traded_notional,
                        'cost_fraction': cost_fraction,
                        'gross_leverage': float(np.abs(target).sum())})
        pre_trade = target * (1 + r) / (1 + gross_return)
    series = pd.DataFrame.from_records(records).set_index('decision_date')
    if predictions is not None:
        if set(predictions.columns) != set(tickers):
            raise ValueError('Prediction columns must match asset universe.')
        if not wdf.index.isin(predictions.index).all():
            raise ValueError('Missing prediction dates.')
        pred = predictions.reindex(index=wdf.index, columns=tickers).astype(float)
        if not np.isfinite(pred.to_numpy()).all():
            raise ValueError('Predictions contain missing/nonfinite values.')
        mse = float(np.mean((pred.to_numpy() - rdf.to_numpy()) ** 2))
    else:
        mse = float('nan')
    rf_weekly = (1 + rf_annual) ** (1 / 52) - 1
    excess = series['net_return'] - rf_weekly
    sd = float(excess.std(ddof=1))
    sharpe = float(np.sqrt(52) * excess.mean() / sd) if sd > 0 else float('nan')
    wealth = pd.concat([pd.Series([1.0], index=[wdf.index[0] - pd.Timedelta(days=7)]),
                        (1 + series['net_return']).cumprod()]).sort_index()
    max_dd = float((wealth / wealth.cummax() - 1).min())
    n = len(series)
    summary = {
        'n_weeks': n,
        'ann_return': float((1 + series['net_return']).prod() ** (52 / n) - 1),
        'ann_volatility': float(series['net_return'].std(ddof=1) * np.sqrt(52)),
        'sharpe_net': sharpe,
        'max_drawdown': max_dd,
        'mean_weekly_turnover': float(series['turnover'].mean()),
        'total_traded_notional': float(series['turnover'].sum()),
        'total_cost_fraction_sum': float(series['cost_fraction'].sum()),
        'prediction_mse': mse,
        'cost_bps': cost_bps,
    }
    return summary, series, wealth


# ---------------- 1. forecasting model ----------------
class PooledLinearForecaster:
    """One pooled linear model across all 20 ETFs. alpha=0 -> OLS, alpha>0 -> ridge.
    Features are z-scored with mean/std from the fitting window only; intercept not penalised."""
    def __init__(self, alpha=0.0):
        self.alpha = float(alpha)

    def fit(self, panel, dates):
        d = panel.loc[panel.index.get_level_values(0).isin(dates)]
        X = d[FEATURES].to_numpy(float); y = d['target_return'].to_numpy(float)
        self.mu_, self.sd_ = X.mean(0), X.std(0, ddof=0)
        Z = (X - self.mu_) / self.sd_
        self.b0_ = y.mean()
        yc = y - self.b0_
        self.coef_ = np.linalg.solve(Z.T @ Z + self.alpha * np.eye(Z.shape[1]), Z.T @ yc)
        return self

    def predict(self, panel, dates):
        d = panel.loc[panel.index.get_level_values(0).isin(dates)]
        Z = (d[FEATURES].to_numpy(float) - self.mu_) / self.sd_
        pred = pd.Series(self.b0_ + Z @ self.coef_, index=d.index)
        return pred.unstack('ticker')[TICKERS]


# ---------------- 2. covariance estimate (past information only) ----------------
def rolling_cov(past_returns, date, window=COV_WINDOW):
    hist = past_returns.loc[:date].tail(window)  # includes week ending at t, never t+1
    if len(hist) < 52:
        raise ValueError(f'Not enough history for covariance at {date}')
    return hist.cov().to_numpy()


# ---------------- 3. constrained mean-variance optimiser ----------------
def mv_weights(mu, Sigma, gamma, w0=None, max_gross=MAX_GROSS, max_w=MAX_W, allow_short=True):
    """max mu'w - gamma/2 w'Sigma w  s.t. sum w = 1, sum|w| <= max_gross, |w_i| <= max_w.
    Split w = u - v with u, v >= 0 so the L1 constraint is linear (exact at optimum)."""
    n = len(mu); S = 1e3  # scale objective for solver tolerance
    if not allow_short:
        bounds = [(0, max_w)] * n + [(0, 0)] * n
    else:
        bounds = [(0, max_w)] * (2 * n)

    def f(x):
        w = x[:n] - x[n:]
        return -S * (mu @ w - 0.5 * gamma * w @ Sigma @ w)

    def g(x):
        w = x[:n] - x[n:]
        gw = -S * (mu - gamma * Sigma @ w)
        return np.concatenate([gw, -gw])

    cons = [{'type': 'eq', 'fun': lambda x: x[:n].sum() - x[n:].sum() - 1.0,
             'jac': lambda x: np.concatenate([np.ones(n), -np.ones(n)])},
            {'type': 'ineq', 'fun': lambda x: max_gross - x.sum(),
             'jac': lambda x: -np.ones(2 * n)}]
    if w0 is None:
        w0 = np.full(n, 1.0 / n)
    x0 = np.concatenate([np.clip(w0, 0, None), np.clip(-w0, 0, None)])
    res = minimize(f, x0, jac=g, bounds=bounds, constraints=cons, method='SLSQP',
                   options={'maxiter': 500, 'ftol': 1e-12})
    w = res.x[:n] - res.x[n:]
    # tidy numerical noise so the strict backtester checks pass
    w[np.abs(w) < 1e-9] = 0.0
    w = np.clip(w, -max_w, max_w)
    w = w + (1.0 - w.sum()) / n
    if np.abs(w).sum() > max_gross:  # rare: rescale shorts slightly
        neg = w < 0; excess = np.abs(w).sum() - max_gross
        w[neg] *= 1 - excess / (2 * np.abs(w[neg]).sum())
        w[~neg] *= (1 - w[neg].sum()) / w[~neg].sum()
    return w, res.success


def build_weights(pred, past_returns, dates, gamma, **kw):
    rows, w_prev, fails = [], None, 0
    for t in dates:
        Sigma = rolling_cov(past_returns, t)
        w, ok = mv_weights(pred.loc[t, TICKERS].to_numpy(), Sigma, gamma, w0=w_prev, **kw)
        fails += (not ok); rows.append(w); w_prev = w
    return pd.DataFrame(rows, index=dates, columns=TICKERS), fails


def run_strategy(pred, targets, past_returns, dates, gamma, allow_short=True):
    W, fails = build_weights(pred, past_returns, dates, gamma, allow_short=allow_short)
    s, series, wealth = evaluate_portfolio(W, targets.loc[dates, TICKERS],
                                           predictions=pred.loc[dates, TICKERS],
                                           cost_bps=COST_BPS, allow_short=allow_short,
                                           max_gross=MAX_GROSS, max_weight_abs=MAX_W)
    s['solver_failures'] = fails
    return s, series, wealth, W


def r2_oos(pred, actual, benchmark=0.0):
    p = pred.to_numpy().ravel(); a = actual.to_numpy().ravel()
    return 1 - np.sum((a - p) ** 2) / np.sum((a - benchmark) ** 2)


In [8]:
panel, SPLIT, TARGETS, PAST = load_panels('train_panel.csv', 'validation_panel.csv')
print({k: len(v) for k, v in SPLIT.items()}, 'weeks')  # expect train 470, validation 157


{'train': 470, 'validation': 157} weeks


## 2. Tuning on the four internal folds (2016–2018 only)

In [9]:
import time, numpy as np, pandas as pd, itertools
tr = SPLIT['train']
# fold schedule: replicate Person 1 (156w train, 26w val, step 26, last 4) on decision dates
n=len(tr); folds=[]
for e in range(156, n-26+1, 26): folds.append((tr[e-156:e], tr[e:e+26]))
folds=folds[-4:]
for i,(a,b) in enumerate(folds,1):
    print(i, (a[0]+pd.Timedelta(days=7)).date(), (a[-1]+pd.Timedelta(days=7)).date(), (b[0]+pd.Timedelta(days=7)).date(), (b[-1]+pd.Timedelta(days=7)).date())
alphas=[0,10,100,1000,3000,10000,30000,100000]
gammas=[1,2.5,5,10,25,50,100]
rows=[]; t0=time.time()
for a in alphas:
    for k,(ftr,fva) in enumerate(folds,1):
        m=PooledLinearForecaster(a).fit(panel, ftr)
        pred=m.predict(panel, fva)
        act=TARGETS.loc[fva]
        mse=float(((pred-act)**2).to_numpy().mean())
        for g in gammas:
            s,_,_,W=run_strategy(pred, TARGETS, PAST, fva, g)
            rows.append(dict(alpha=a,gamma=g,fold=k,mse=mse,r2_zero=r2_oos(pred,act),sharpe=s['sharpe_net'],
                             turnover=s['mean_weekly_turnover'],fails=s['solver_failures']))
    print(a, round(time.time()-t0,1))
R=pd.DataFrame(rows); R.to_csv('tuning_folds.csv',index=False)
print(R.groupby('alpha')[['mse','r2_zero']].mean())
print(R.groupby(['alpha','gamma'])['sharpe'].mean().unstack().round(2))
print('fails', R.fails.sum())


1 2013-12-27 2016-12-16 2016-12-23 2017-06-16
2 2014-06-27 2017-06-16 2017-06-23 2017-12-15
3 2014-12-26 2017-12-15 2017-12-22 2018-06-15
4 2015-06-26 2018-06-15 2018-06-22 2018-12-14
0 16.7
10 33.6
100 50.3
1000 70.5
3000 88.7
10000 107.0
30000 129.7
100000 149.8
             mse   r2_zero
alpha                     
0       0.000329 -0.014956
10      0.000329 -0.014771
100     0.000328 -0.013205
1000    0.000327 -0.003413
3000    0.000326  0.004866
10000   0.000325  0.011927
30000   0.000325  0.014771
100000  0.000326  0.015550
gamma   1.0    2.5    5.0    10.0   25.0   50.0   100.0
alpha                                                  
0        0.26   0.31   0.30   0.49   0.55   0.67   0.85
10       0.27   0.31   0.30   0.49   0.56   0.67   0.85
100      0.29   0.31   0.29   0.51   0.58   0.69   0.87
1000     0.46   0.50   0.71   0.72   0.80   0.88   0.98
3000     0.83   0.88   1.03   1.01   0.96   1.05   1.04
10000    0.67   1.17   1.20   1.06   1.07   1.13   1.04
30000    1.00   1

## 3. Final fit on 2010–2018, evaluated on 2019–2021 validation, plus references and diagnostics
The α × γ grid on validation is a **diagnostic** of how far MSE and Sharpe diverge. It is not used to pick the model.

In [10]:
import numpy as np, pandas as pd, json
from scipy.stats import spearmanr
tr, va = SPLIT['train'], SPLIT['validation']
act = TARGETS.loc[va, TICKERS]
ALPHA, GAMMA = 10000, 5.0          # chosen on internal folds (MSE for alpha, Sharpe for gamma)
OLS_GAMMA = 100.0                  # best fold Sharpe for OLS (grid edge)

res, store = {}, {}
def record(name, pred, gamma, allow_short=True):
    s, series, wealth, W = run_strategy(pred, TARGETS, PAST, va, gamma, allow_short)
    s['r2_vs_zero'] = r2_oos(pred, act); s['gamma']=gamma
    ic = [spearmanr(pred.loc[t], act.loc[t]).statistic for t in va]
    s['mean_rank_ic'] = float(np.nanmean(ic))
    s['gross_sharpe'] = float(np.sqrt(52)*series.gross_return.mean()/series.gross_return.std())
    s['mean_gross_leverage'] = float(series.gross_leverage.mean())
    s['share_weeks_at_lev_cap'] = float((series.gross_leverage>1.5-1e-3).mean())
    s['mean_short_exposure'] = float(W.clip(upper=0).sum(1).abs().mean())
    s['mean_eff_n'] = float((1/(W**2).sum(1)).mean())
    s['mean_max_abs_w'] = float(W.abs().max(1).mean())
    res[name]=s; store[name]=dict(series=series, wealth=wealth, W=W, pred=pred)

ridge = PooledLinearForecaster(ALPHA).fit(panel, tr); P_r = ridge.predict(panel, va)
ols = PooledLinearForecaster(0).fit(panel, tr);       P_o = ols.predict(panel, va)
hist = pd.DataFrame(np.tile(TARGETS.loc[tr, TICKERS].mean().to_numpy(), (len(va),1)), index=va, columns=TICKERS)
zero = pd.DataFrame(0.0, index=va, columns=TICKERS)
record('Ridge + MV (baseline)', P_r, GAMMA)
record('OLS + MV', P_o, OLS_GAMMA)
record('OLS + MV (gamma=5)', P_o, GAMMA)
record('Hist. mean + MV', hist, GAMMA)
record('Zero forecast = min-variance', zero, GAMMA)

bm = pd.read_csv('validation_benchmark_metrics.csv').set_index('strategy')
T = pd.DataFrame(res).T
T.to_csv('validation_results_person2.csv')
cols=['sharpe_net','gross_sharpe','ann_return','ann_volatility','max_drawdown','mean_weekly_turnover',
      'total_cost_fraction_sum','prediction_mse','r2_vs_zero','mean_rank_ic','mean_gross_leverage',
      'share_weeks_at_lev_cap','mean_short_exposure','mean_eff_n','solver_failures']
pd.set_option('display.width',250)
print(T[cols].astype(float).round(4).T)
print(bm[['sharpe_net','ann_return','ann_volatility','max_drawdown','mean_weekly_turnover']].round(4))
print('ridge coef (per 1 sd):', dict(zip(FEATURES, np.round(ridge.coef_*1e4,2))), 'b0bp', ridge.b0_*1e4)
print('ols coef (per 1 sd):', dict(zip(FEATURES, np.round(ols.coef_*1e4,2))))
print('pred cross-sec sd (bp): ridge', (P_r.std(1).mean()*1e4).round(2), 'ols', (P_o.std(1).mean()*1e4).round(2), 'actual', (act.std(1).mean()*1e4).round(1))

# cost sensitivity for baseline weights
W = store['Ridge + MV (baseline)']['W']
cs={}
for c in [0,2,5,10,20]:
    s,_,_ = evaluate_portfolio(W, act, cost_bps=c)
    cs[c]=s['sharpe_net']
print('cost sens', {k:round(v,3) for k,v in cs.items()})
# grid on validation: MSE vs Sharpe (diagnostic only, not selection)
grid=[]
for a in [0,100,1000,3000,10000,30000,100000]:
    P = PooledLinearForecaster(a).fit(panel,tr).predict(panel,va)
    for g in [1,2.5,5,10,25,50,100]:
        s,_,_,_ = run_strategy(P, TARGETS, PAST, va, g)
        grid.append(dict(alpha=a,gamma=g,mse=s['prediction_mse'],sharpe=s['sharpe_net'],turnover=s['mean_weekly_turnover']))
G=pd.DataFrame(grid); G.to_csv('validation_mse_vs_sharpe_grid.csv',index=False)
print(G.pivot(index='alpha',columns='gamma',values='sharpe').round(2))
print(G.groupby('alpha').mse.first().map(lambda v:f'{v:.8f}'))
import pickle; pickle.dump(dict(store=store, cs=cs, coef=dict(ridge=ridge.coef_, ols=ols.coef_)), open('store.pkl','wb'))


/tmp/ipykernel_2703/958167359.py:12: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  ic = [spearmanr(pred.loc[t], act.loc[t]).statistic for t in va]
/tmp/ipykernel_2703/958167359.py:13: RuntimeWarning: Mean of empty slice
  s['mean_rank_ic'] = float(np.nanmean(ic))


                         Ridge + MV (baseline)  OLS + MV  OLS + MV (gamma=5)  Hist. mean + MV  Zero forecast = min-variance
sharpe_net                              0.4371    0.6754              0.3971           1.4551                        0.8920
gross_sharpe                            0.5940    0.8668              0.5728           1.4653                        0.9141
ann_return                              0.0885    0.0506              0.0751           0.2520                        0.0334
ann_volatility                          0.3013    0.0776              0.2919           0.1640                        0.0376
max_drawdown                           -0.5033   -0.1256             -0.4821          -0.2132                       -0.0580
mean_weekly_turnover                    1.8223    0.5737              1.9754           0.0667                        0.0329
total_cost_fraction_sum                 0.1430    0.0450              0.1551           0.0052                        0.0026
predicti

## 4. Figures used in the slides

In [11]:
import pickle, numpy as np, pandas as pd, matplotlib
matplotlib.use('Agg'); import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm, LinearSegmentedColormap
d=pickle.load(open('store.pkl','rb')); S=d['store']
T=pd.read_csv('validation_results_person2.csv',index_col=0)
bm=pd.read_csv('validation_benchmark_metrics.csv').set_index('strategy')
G=pd.read_csv('validation_mse_vs_sharpe_grid.csv')
BLUE,ORANGE,AQUA,INK,MUTED,GRID='#2a78d6','#eb6834','#1baf7a','#0b0b0b','#52514e','#e4e3df'
plt.rcParams.update({'font.size':12,'axes.edgecolor':MUTED,'axes.labelcolor':INK,'xtick.color':MUTED,'ytick.color':MUTED,
  'axes.spines.top':False,'axes.spines.right':False,'axes.grid':True,'grid.color':GRID,'grid.linewidth':0.8,'axes.axisbelow':True,
  'figure.dpi':200,'savefig.bbox':'tight','legend.frameon':False})

# F1 MSE vs Sharpe
fig,ax=plt.subplots(figsize=(8,4.6))
ax.scatter(G.mse*1e4,G.sharpe,s=22,color='#b9b8b2',label='Ridge/OLS × risk-aversion grid (49 runs)',zorder=2)
pts={'Ridge + MV (baseline)':(BLUE,'Ridge + MV\n(our baseline)'),'Hist. mean + MV':(ORANGE,'Historical mean + MV'),
     'Zero forecast = min-variance':(AQUA,'Zero forecast\n(= min-variance)')}
for k,(c,l) in pts.items():
    r=T.loc[k]; ax.scatter(r.prediction_mse*1e4,r.sharpe_net,s=110,color=c,edgecolor='white',linewidth=2,zorder=4)
    ax.annotate(l,(r.prediction_mse*1e4,r.sharpe_net),xytext=(10,-4),textcoords='offset points',fontsize=11,color=INK,va='center')
for name,ls in [('SPY buy-and-hold','--'),('Weekly equal-weight',':')]:
    v=bm.loc[name,'sharpe_net']; ax.axhline(v,color=MUTED,ls=ls,lw=1.2)
    ax.text(10.07,v+0.03,f'{name} {v:.2f}',color=MUTED,fontsize=10,ha='right')
ax.set_xlabel('Forecast MSE on validation (×10⁻⁴)  ← more accurate'); ax.set_ylabel('Net Sharpe (validation)')
ax.set_xlim(9.62,10.1); ax.set_ylim(0,1.65); ax.legend(loc='lower left',fontsize=10)
fig.savefig('fig/f1_mse_vs_sharpe.png'); plt.close()

# F2 weights heatmap
W=S['Ridge + MV (baseline)']['W']
cmap=LinearSegmentedColormap.from_list('d',['#c2410c','#f3f2ee','#1d5fb0'])
fig,ax=plt.subplots(figsize=(10,5))
im=ax.imshow(W.T.values,aspect='auto',cmap=cmap,norm=TwoSlopeNorm(0,-0.5,1.0),interpolation='nearest')
ax.set_yticks(range(20),TICKERS,fontsize=9); ax.grid(False)
yrs=[i for i,t in enumerate(W.index) if i>0 and t.year!=W.index[i-1].year]
ax.set_xticks(yrs,[str(W.index[i].year) for i in yrs])
cb=fig.colorbar(im,ax=ax,fraction=0.03,pad=0.01); cb.set_label('Portfolio weight'); cb.outline.set_visible(False)
ax.set_title('Ridge + MV weights jump between corner positions week to week',loc='left',fontsize=14,color=INK)
fig.savefig('fig/f2_weights_heatmap.png'); plt.close()

# F3 wealth
fig,ax=plt.subplots(figsize=(8,4.6))
b=S['Ridge + MV (baseline)']; ser=b['series']
gross=pd.concat([pd.Series([1.0],index=[ser.index[0]-pd.Timedelta(days=7)]),(1+ser.gross_return).cumprod()])
ax.plot(gross.index,gross.values,color=BLUE,lw=1.5,ls='--',label='Ridge + MV, before costs')
ax.plot(b['wealth'].index,b['wealth'].values,color=BLUE,lw=2.2,label='Ridge + MV, net of 5 bp costs')
mv=S['Zero forecast = min-variance']['wealth']; ax.plot(mv.index,mv.values,color=AQUA,lw=2,label='Min-variance (no forecast)')
_,SP,TG,_=load_panels(); va=SP['validation']
for nm,alloc,ls in [('SPY buy-and-hold',{'SPY':1.0},'--'),('Weekly equal-weight',{t:0.05 for t in TICKERS},':')]:
    w=pd.DataFrame(0.0,index=va,columns=TICKERS)
    for k,v in alloc.items(): w[k]=v
    _,_,wl=evaluate_portfolio(w,TG.loc[va,TICKERS],allow_short=False,max_gross=1.0)
    ax.plot(wl.index,wl.values,color=MUTED,lw=1.4,ls=ls,label=nm)
import matplotlib.dates as mdates; ax.xaxis.set_major_locator(mdates.YearLocator()); ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y')); ax.set_ylabel('Wealth (starts at 1)'); ax.legend(loc='upper left',fontsize=10)
fig.savefig('fig/f3_wealth.png'); plt.close()

# F4 cost sensitivity + turnover
cs=d['cs']; fig,ax=plt.subplots(figsize=(6,4.2))
xs=list(cs.keys()); ys=list(cs.values())
ax.bar([str(x) for x in xs],ys,color=[BLUE if x!=5 else '#1d5fb0' for x in xs],width=0.6)
for x,y in zip(xs,ys): ax.text(str(x),(y+0.02) if y>=0 else 0.02,f'{y:.2f}',ha='center',fontsize=11,color=INK)
ax.set_ylim(-0.1,0.68); ax.axhline(0,color=MUTED,lw=1); ax.set_xlabel('Cost per dollar traded (bp)'); ax.set_ylabel('Net Sharpe, same weights')
ax.set_title('Cost sensitivity at 182% weekly turnover',loc='left',fontsize=13,color=INK); ax.grid(axis='x',visible=False)
fig.savefig('fig/f4_cost_sensitivity.png'); plt.close()
print('ok')


ok


## 5. Update the shared iteration log

In [12]:
# Append Person 2 rows to the shared iteration log
import csv, datetime as dt
today = dt.date.today().isoformat()
T = pd.read_csv('validation_results_person2.csv', index_col=0)
rows = [
 ['1a', today, 'Person 2', 'Forecast: pooled ridge (alpha=10000, chosen by fold MSE) + MV (gamma=5, chosen by fold Sharpe)',
  'An accuracy-trained forecast plus MV gives a usable portfolio', '2019-2021',
  round(T.loc['Ridge + MV (baseline)','sharpe_net'],4), f"{T.loc['Ridge + MV (baseline)','prediction_mse']:.6e}",
  round(T.loc['Ridge + MV (baseline)','mean_weekly_turnover'],4), 'Keep as baseline',
  'Below both benchmarks and min-variance; leverage cap binds every week; eff. N ~1.5; 14% cost drag'],
 ['1b', today, 'Person 2', 'OLS (alpha=0) + MV, gamma=100 (fold-best, grid edge)', 'Less shrinkage, more risk aversion', '2019-2021',
  round(T.loc['OLS + MV','sharpe_net'],4), f"{T.loc['OLS + MV','prediction_mse']:.6e}", round(T.loc['OLS + MV','mean_weekly_turnover'],4),
  'Reference', 'Higher Sharpe mainly via lower turnover (high gamma), not better forecasts'],
 ['1c', today, 'Person 2', 'Reference: training historical-mean forecast + MV (gamma=5)', 'Does forecast accuracy matter for the decision?', '2019-2021',
  round(T.loc['Hist. mean + MV','sharpe_net'],4), f"{T.loc['Hist. mean + MV','prediction_mse']:.6e}", round(T.loc['Hist. mean + MV','mean_weekly_turnover'],4),
  'Diagnostic only', 'Worse MSE, much better Sharpe; not pre-registered, partly a 2019-21 equity-market effect'],
 ['1d', today, 'Person 2', 'Reference: zero forecast = constrained min-variance', 'Value of any forecast', '2019-2021',
  round(T.loc['Zero forecast = min-variance','sharpe_net'],4), f"{T.loc['Zero forecast = min-variance','prediction_mse']:.6e}",
  round(T.loc['Zero forecast = min-variance','mean_weekly_turnover'],4), 'Diagnostic only', 'Beats the ridge baseline with no forecast at all'],
]
with open('iteration_log.csv','a',newline='') as f: csv.writer(f).writerows(rows)
print(pd.read_csv('iteration_log.csv').to_string())


     iteration      date                                                                                           owner                                              changed_component hypothesis  validation_period validation_sharpe_net validation_mse          turnover                                                                                           decision  notes
0          NaN      Team                                                                                        Protocol                      Freeze data/splits/metrics before fitting        NaN                NaN                                                                                                                                                   Pending    NaN
1a  2026-11-06  Person 2  Forecast: pooled ridge (alpha=10000, chosen by fold MSE) + MV (gamma=5, chosen by fold Sharpe)  An accuracy-trained forecast plus MV gives a usable portfolio  2019-2021             0.4371          9.716593e-04         1.8223

# 6. Experiment

Experiment 1: Distributionally robust optimisation

In [13]:
# EXPERIMENT: Robust portfolio optimisation
# Paper-inspired uncertainty penalty
# Uses the existing Ridge forecasts and backtester

from scipy.optimize import minimize
import numpy as np
import pandas as pd

def robust_mv_weights(mu, Sigma, gamma, rho=0.001,
                      allow_short=True):
    n = len(mu)
    scale = 1000.0

    bounds = (
        [(0, MAX_W)] * (2*n)
        if allow_short
        else [(0, MAX_W)] * n + [(0, 0)] * n
    )

    def objective(x):
        w = x[:n] - x[n:]
        utility = (
            mu @ w
            - 0.5 * gamma * w @ Sigma @ w
            - rho * np.linalg.norm(w)
        )
        return -scale * utility

    def gradient(x):
        w = x[:n] - x[n:]
        norm = max(np.linalg.norm(w), 1e-10)
        grad_w = -scale * (
            mu - gamma * Sigma @ w - rho*w/norm
        )
        return np.r_[grad_w, -grad_w]

    constraints = [
        {
            'type': 'eq',
            'fun': lambda x: x[:n].sum()-x[n:].sum()-1
        },
        {
            'type': 'ineq',
            'fun': lambda x: MAX_GROSS-x.sum()
        }
    ]

    x0 = np.r_[np.ones(n)/n, np.zeros(n)]

    result = minimize(
        objective, x0, jac=gradient,
        bounds=bounds,
        constraints=constraints,
        method='SLSQP',
        options={'maxiter':500, 'ftol':1e-10}
    )

    if not result.success:
        raise RuntimeError(result.message)

    w = result.x[:n]-result.x[n:]

    return w


def run_robust(rho):
    weights = []

    for date in va:
        Sigma = rolling_cov(PAST, date)
        mu = P_r.loc[date, TICKERS].to_numpy()

        w = robust_mv_weights(
            mu, Sigma, GAMMA, rho=rho
        )
        weights.append(w)

    W = pd.DataFrame(
        weights, index=va, columns=TICKERS
    )

    metrics, series, wealth = evaluate_portfolio(
        W,
        TARGETS.loc[va, TICKERS],
        predictions=P_r.loc[va, TICKERS],
        cost_bps=COST_BPS
    )

    return metrics, series, wealth, W


# Predefined sensitivity values
rho_values = [0, 0.0001, 0.0005, 0.001, 0.005]

rows = []
robust_outputs = {}

for rho in rho_values:
    metrics, series, wealth, weights = run_robust(rho)
    robust_outputs[rho] = (metrics, series, wealth, weights)

    rows.append({
        "rho": rho,
        "sharpe": metrics["sharpe_net"],
        "turnover": metrics["mean_weekly_turnover"],
        "drawdown": metrics["max_drawdown"],
        "annual_return": metrics["ann_return"]
    })

robust_results = pd.DataFrame(rows)

print(robust_results.round(4).to_string(index=False))

robust_results.to_csv(
    "robust_experiment_results.csv",
    index=False
)


   rho  sharpe  turnover  drawdown  annual_return
0.0000  0.4371    1.8223   -0.5033         0.0885
0.0001  0.4390    1.7910   -0.5027         0.0890
0.0005  0.4470    1.6619   -0.4984         0.0911
0.0010  0.4752    1.5499   -0.4876         0.0992
0.0050  0.6771    0.7737   -0.4025         0.1461


Experiment 2: Robust Optimisation + transaction cost penalty

In [ ]:
# EXPERIMENT 2: Robust optimisation + transaction-cost penalty

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize

RHO = 0.005
LAMBDAS = [0, 0.0001, 0.0005, 0.001, 0.002]

def cost_aware_weights(mu, Sigma, gamma, rho, lam, pretrade):
    n = len(mu)
    scale = 1000.0

    def objective(w):
        return -scale * (
            mu @ w
            - 0.5 * gamma * (w @ Sigma @ w)
            - rho * np.linalg.norm(w)
            - lam * np.sqrt((w - pretrade)**2 + 1e-12).sum()
        )

    def gradient(w):
        norm = max(np.linalg.norm(w), 1e-12)
        diff = w - pretrade

        return -scale * (
            mu - gamma * Sigma @ w
            - rho * w / norm
            - lam * diff / np.sqrt(diff**2 + 1e-12)
        )

    # Long-short, with gross leverage <= 1.5
    # Auxiliary variables u represent absolute positions.
    x0 = np.r_[np.ones(n)/n, np.ones(n)/n]

    def f(x):
        return objective(x[:n])

    def jac(x):
        return np.r_[gradient(x[:n]), np.zeros(n)]

    cons = [
        {'type':'eq',
         'fun':lambda x: x[:n].sum()-1},
        {'type':'ineq',
         'fun':lambda x: x[n:]-x[:n]},
        {'type':'ineq',
         'fun':lambda x: x[n:]+x[:n]},
        {'type':'ineq',
         'fun':lambda x: MAX_GROSS-x[n:].sum()}
    ]

    bounds = [(-MAX_W, MAX_W)]*n + [(0, MAX_W)]*n

    res = minimize(
        f, x0, jac=jac, bounds=bounds,
        constraints=cons, method='SLSQP',
        options={'maxiter':500, 'ftol':1e-10}
    )

    if not res.success:
        raise RuntimeError(res.message)

    w = res.x[:n]

    if (abs(w.sum()-1)>1e-6 or
        np.abs(w).sum()>MAX_GROSS+1e-6):
        raise ValueError("Constraint violation")

    return w


def run_cost_aware(lam):
    weights = []
    pretrade = np.zeros(len(TICKERS))

    for date in va:
        Sigma = rolling_cov(PAST, date)
        mu = P_r.loc[date, TICKERS].to_numpy()

        w = cost_aware_weights(
            mu, Sigma, GAMMA, RHO, lam, pretrade
        )

        weights.append(w)

        # Drift holdings after next-week realised returns
        r = TARGETS.loc[date, TICKERS].to_numpy()
        growth = 1 + np.dot(w, r)

        if growth <= 0:
            raise ValueError("Nonpositive portfolio growth")

        pretrade = w * (1+r) / growth

    W = pd.DataFrame(weights, index=va, columns=TICKERS)

    metrics, series, wealth = evaluate_portfolio(
        W, TARGETS.loc[va, TICKERS],
        predictions=P_r.loc[va, TICKERS],
        cost_bps=COST_BPS
    )

    return metrics, series, wealth, W


rows = []
cost_outputs = {}

for lam in LAMBDAS:
    m, s, wealth, W = run_cost_aware(lam)
    cost_outputs[lam] = (m, s, wealth, W)

    rows.append({
        'lambda': lam,
        'sharpe': m['sharpe_net'],
        'turnover': m['mean_weekly_turnover'],
        'drawdown': m['max_drawdown'],
        'annual_return': m['ann_return']
    })

cost_results = pd.DataFrame(rows)

print(cost_results.round(4).to_string(index=False))
cost_results.to_csv('cost_aware_results.csv', index=False)

# Visual comparison
plt.figure(figsize=(9, 5))
for lam in LAMBDAS:
    plt.plot(
        cost_outputs[lam][2].index,
        cost_outputs[lam][2].values,
        label=f'lambda={lam}'
    )

plt.title('Robust + transaction-cost-aware portfolios')
plt.ylabel('Cumulative net wealth')
plt.legend()
plt.grid(alpha=0.3)
plt.savefig('cost_aware_wealth.png', dpi=200, bbox_inches='tight')
plt.show()


Experiment 3: end-to-end decision-aware learning

In [ ]:

# EXPERIMENT 3: Decision-aware end-to-end learning
# Paper-inspired, simplified differentiable portfolio layer

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)
torch.set_num_threads(2)

tr = SPLIT["train"]
va = SPLIT["validation"]
n = len(TICKERS)

# All preprocessing fitted on training data only
train_df = panel.loc[
    panel.index.get_level_values(0).isin(tr)
]
mean = train_df[FEATURES].mean().to_numpy(float)
std = train_df[FEATURES].std(ddof=0).to_numpy(float)
std = np.where(std < 1e-10, 1, std)

def make_data(dates):
    X, Y, C = [], [], []
    for date in dates:
        x = panel.loc[date].reindex(TICKERS)[FEATURES].to_numpy(float)
        X.append((x - mean) / std)
        Y.append(TARGETS.loc[date, TICKERS].to_numpy(float))
        C.append(rolling_cov(PAST, date))
    return (
        torch.tensor(np.array(X), dtype=torch.float32),
        torch.tensor(np.array(Y), dtype=torch.float32),
        torch.tensor(np.array(C), dtype=torch.float32)
    )


# Only use training dates with sufficient covariance history
# Preserve the original 52-week minimum

valid_train_dates = []

for date in tr:
    try:
        rolling_cov(PAST, date)
        valid_train_dates.append(date)
    except ValueError:
        continue

print("Original training weeks:", len(tr))
print("Usable training weeks:", len(valid_train_dates))
print("Excluded early weeks:", len(tr) - len(valid_train_dates))

# Rebuild training tensors using valid dates only
tr = pd.DatetimeIndex(valid_train_dates)

Xtr, Ytr, Ctr = make_data(tr)
Xva, Yva, Cva = make_data(va)

print("Training tensors:", Xtr.shape, Ytr.shape, Ctr.shape)
print("Validation tensors:", Xva.shape, Yva.shape, Cva.shape)

# A shared forecasting architecture
# Each ETF receives a forecast from the same linear model
class PortfolioModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = torch.nn.Linear(len(FEATURES), 1)

    def forward(self, X):
        return self.linear(X).squeeze(-1)

# Differentiable portfolio construction
# Long-only, fully invested
TEMPERATURE = 0.002
RISK_GAMMA = 5.0
COST = 0.0005

def portfolio_weights(pred):
    return torch.softmax(pred / TEMPERATURE, dim=1)

def decision_loss(model, X, Y, C):
    pred = model(X)
    w = portfolio_weights(pred)

    gross = (w * Y).sum(dim=1)
    risk = torch.einsum("bi,bij,bj->b", w, C, w)

    # Differentiable turnover approximation
    # First observation assumes deployment from cash
    previous = torch.cat(
        [torch.zeros_like(w[:1]), w[:-1]], dim=0
    )
    turnover = torch.sqrt(
        (w - previous)**2 + 1e-10
    ).sum(dim=1)

    utility = (
        gross
        - 0.5 * RISK_GAMMA * risk
        - COST * turnover
    )

    return -utility.mean()

# Internal holdout: last 26 training weeks
cut = len(tr) - 26

def fit_model(mode, epochs, full=False):
    model = PortfolioModel()

    # Start from zero forecasts -> equal weights
    with torch.no_grad():
        model.linear.weight.zero_()
        model.linear.bias.zero_()

    optimizer = torch.optim.Adam(
        model.parameters(), lr=0.0001
    )

    end = len(tr) if full else cut

    for epoch in range(epochs):
        optimizer.zero_grad()

        if mode == "MSE":
            loss = F.mse_loss(
                model(Xtr[:end]), Ytr[:end]
            )
        else:
            loss = decision_loss(
                model, Xtr[:end],
                Ytr[:end], Ctr[:end]
            )

        loss.backward()
        optimizer.step()

    return model

# Select training duration without touching 2019–2021
EPOCHS = [25, 50, 100, 200]
selection = []

for mode in ["MSE", "Decision-aware"]:
    for epochs in EPOCHS:
        model = fit_model(mode, epochs)
        with torch.no_grad():
            score = -decision_loss(
                model,
                Xtr[cut:], Ytr[cut:], Ctr[cut:]
            ).item()

        selection.append({
            "model": mode,
            "epochs": epochs,
            "holdout_utility": score
        })

selection_df = pd.DataFrame(selection)
print("INTERNAL TRAINING HOLDOUT")
print(selection_df.round(6).to_string(index=False))

# Refit on all 2010–2018 training observations
models = {}
for mode in ["MSE", "Decision-aware"]:
    subset = selection_df[
        selection_df["model"] == mode
    ]
    best_epochs = int(
        subset.loc[
            subset["holdout_utility"].idxmax(),
            "epochs"
        ]
    )
    models[mode] = fit_model(
        mode, best_epochs, full=True
    )
    print(mode, "selected epochs:", best_epochs)

# Evaluate with the ORIGINAL shared backtester
results = []
outputs = {}

for mode, model in models.items():
    with torch.no_grad():
        pred = model(Xva).numpy()
        weights = portfolio_weights(
            model(Xva)
        ).numpy()

    P = pd.DataFrame(
        pred, index=va, columns=TICKERS
    )
    W = pd.DataFrame(
        weights, index=va, columns=TICKERS
    )

    metrics, series, wealth = evaluate_portfolio(
        W,
        TARGETS.loc[va, TICKERS],
        predictions=P,
        cost_bps=COST_BPS,
        allow_short=False,
        max_gross=1.0
    )

    results.append({
        "model": mode,
        "sharpe": metrics["sharpe_net"],
        "mse": metrics["prediction_mse"],
        "turnover": metrics["mean_weekly_turnover"],
        "drawdown": metrics["max_drawdown"],
        "annual_return": metrics["ann_return"]
    })

    outputs[mode] = {
        "pred": P,
        "weights": W,
        "wealth": wealth,
        "metrics": metrics
    }

results3 = pd.DataFrame(results)
print("\nEXPERIMENT 3 RESULTS")
print(results3.round(6).to_string(index=False))

# Wealth comparison
plt.figure(figsize=(10, 5))
for mode in outputs:
    wealth = outputs[mode]["wealth"]
    plt.plot(
        wealth.index, wealth.values,
        label=mode, linewidth=2
    )

plt.title("MSE training vs decision-aware training")
plt.xlabel("Date")
plt.ylabel("Cumulative net wealth")
plt.legend()
plt.grid(alpha=0.3)
plt.savefig(
    "experiment3_end_to_end.png",
    dpi=200, bbox_inches="tight"
)
plt.show()

selection_df.to_csv(
    "experiment3_internal_selection.csv",
    index=False
)
results3.to_csv(
    "experiment3_results.csv",
    index=False
)


Experiment 4: proper rolling-fold hyperparameter tuning

In [ ]:
# EXPERIMENT 4: Training-only rolling-fold tuning
# Run AFTER Experiments 1 and 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

assert "cost_aware_weights" in globals()
assert "folds" in globals()

# Fixed Ridge alpha and risk aversion from original baseline
TUNE_ALPHA = 10000
TUNE_GAMMA = 5.0

RHO_GRID = [0.0, 0.001, 0.005]
LAMBDA_GRID = [0.0, 0.0005, 0.002]

def backtest_robust(pred, dates, rho, lam):
    """Build sequential portfolio weights and evaluate
    using the original shared backtester."""

    dates = pd.DatetimeIndex(dates)
    weights = []
    pretrade = np.zeros(len(TICKERS))

    for date in dates:
        Sigma = rolling_cov(PAST, date)
        mu = pred.loc[date, TICKERS].to_numpy(float)

        w = cost_aware_weights(
            mu, Sigma, TUNE_GAMMA,
            rho, lam, pretrade
        )

        weights.append(w)

        # Drift portfolio using realised next-week returns.
        # These returns only affect the following decision.
        r = TARGETS.loc[date, TICKERS].to_numpy(float)
        growth = 1.0 + np.dot(w, r)

        if growth <= 0:
            raise ValueError("Invalid portfolio growth")

        pretrade = w * (1.0 + r) / growth

    W = pd.DataFrame(
        weights, index=dates, columns=TICKERS
    )

    m, series, wealth = evaluate_portfolio(
        W,
        TARGETS.loc[dates, TICKERS],
        predictions=pred.loc[dates, TICKERS],
        cost_bps=COST_BPS,
        allow_short=True,
        max_gross=MAX_GROSS,
        max_weight_abs=MAX_W
    )

    return m, series, wealth, W


# Precompute fold-specific forecasts
fold_predictions = []

for fold_id, (train_dates, val_dates) in enumerate(folds, 1):
    model = PooledLinearForecaster(
        TUNE_ALPHA
    ).fit(panel, train_dates)

    pred = model.predict(panel, val_dates)

    fold_predictions.append(
        (fold_id, val_dates, pred)
    )

# Tune both penalties
rows = []

for rho in RHO_GRID:
    for lam in LAMBDA_GRID:
        for fold_id, dates, pred in fold_predictions:

            m, series, wealth, W = backtest_robust(
                pred, dates, rho, lam
            )

            rows.append({
                "rho": rho,
                "lambda": lam,
                "fold": fold_id,
                "sharpe": m["sharpe_net"],
                "turnover": m["mean_weekly_turnover"],
                "drawdown": m["max_drawdown"],
                "annual_return": m["ann_return"]
            })

        print("Completed:", rho, lam)

fold_results = pd.DataFrame(rows)

# Aggregate the four folds
summary = (
    fold_results
    .groupby(["rho", "lambda"], as_index=False)
    .agg(
        mean_sharpe=("sharpe", "mean"),
        sd_sharpe=("sharpe", "std"),
        mean_turnover=("turnover", "mean"),
        mean_drawdown=("drawdown", "mean")
    )
)

summary = summary.sort_values(
    ["mean_sharpe", "mean_turnover"],
    ascending=[False, True]
)

print("\nTRAINING-ONLY TUNING RESULTS")
print(summary.round(4).to_string(index=False))

# Select parameters using training folds ONLY
best = summary.iloc[0]

BEST_RHO = float(best["rho"])
BEST_LAMBDA = float(best["lambda"])

print("\nSELECTED PARAMETERS")
print("rho =", BEST_RHO)
print("lambda =", BEST_LAMBDA)

fold_results.to_csv(
    "experiment4_fold_results.csv", index=False
)
summary.to_csv(
    "experiment4_tuning_summary.csv", index=False
)


In [ ]:
# EXPERIMENT 4: Validation evaluation
# No parameter selection from validation results

selected_metrics, selected_series, selected_wealth, selected_W = (
    backtest_robust(
        P_r, va,
        BEST_RHO, BEST_LAMBDA
    )
)

baseline_metrics = res["Ridge + MV (baseline)"]

comparison = pd.DataFrame({
    "Original Ridge": {
        "Net Sharpe": baseline_metrics["sharpe_net"],
        "Annual return": baseline_metrics["ann_return"],
        "Weekly turnover": baseline_metrics["mean_weekly_turnover"],
        "Max drawdown": baseline_metrics["max_drawdown"]
    },
    "Fold-selected robust model": {
        "Net Sharpe": selected_metrics["sharpe_net"],
        "Annual return": selected_metrics["ann_return"],
        "Weekly turnover": selected_metrics["mean_weekly_turnover"],
        "Max drawdown": selected_metrics["max_drawdown"]
    }
})

print("\nVALIDATION COMPARISON")
print(comparison.round(4).to_string())

comparison.to_csv(
    "experiment4_validation_comparison.csv"
)

plt.figure(figsize=(10, 5))

baseline_wealth = store[
    "Ridge + MV (baseline)"
]["wealth"]

plt.plot(
    baseline_wealth.index,
    baseline_wealth.values,
    label="Original Ridge",
    linewidth=2
)

plt.plot(
    selected_wealth.index,
    selected_wealth.values,
    label="Fold-selected robust model",
    linewidth=2
)

plt.title("Training-selected robust model vs Ridge baseline")
plt.xlabel("Date")
plt.ylabel("Cumulative net wealth")
plt.legend()
plt.grid(alpha=0.3)

plt.savefig(
    "experiment4_validation.png",
    dpi=200,
    bbox_inches="tight"
)
plt.show()


In [ ]:

print("SELECTED PARAMETERS")
print("rho:", BEST_RHO)
print("lambda:", BEST_LAMBDA)

print("\nTOP TRAINING-FOLD CONFIGURATIONS")
print(summary.head(9).round(4).to_string(index=False))

print("\nPER-FOLD RESULTS FOR SELECTED MODEL")
selected_folds = fold_results[
    np.isclose(fold_results["rho"], BEST_RHO) &
    np.isclose(fold_results["lambda"], BEST_LAMBDA)
]
print(selected_folds.round(4).to_string(index=False))
